# This nb is the reference implementation for the LSTM Layer in pytorch

This was used to build a unit test for the correct gradients for SystemDS internal CPU operand

In [1]:
import numpy as np
import torch
import torch.nn as nn

In [82]:
torch.set_printoptions(precision=3,sci_mode=False,linewidth=400)

In [83]:
torch.tensor([[0.00012345]], dtype=torch.float64)

tensor([[0.000]], dtype=torch.float64)

In [84]:
# PyTorch LSTM implementation
class CustomLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, reverse=False):
        super(CustomLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True, bidirectional=reverse)
        self.lstm.weight_ih_l0 = nn.Parameter(torch.from_numpy(w_ih))
        self.lstm.weight_hh_l0 = nn.Parameter(torch.from_numpy(w_hh))
        self.lstm.bias_ih_l0 = nn.Parameter(torch.from_numpy(b_ih))
        self.lstm.bias_hh_l0 = nn.Parameter(torch.from_numpy(b_hh))
        
        if reverse:
            self.lstm.weight_ih_l0_reverse.data = torch.from_numpy(w_ih)
            self.lstm.weight_hh_l0_reverse.data = torch.from_numpy(w_hh)
            self.lstm.bias_ih_l0_reverse.data = torch.from_numpy(b_ih)
            self.lstm.bias_hh_l0_reverse.data = torch.from_numpy(b_hh)

    def forward(self, x, hc):
        return self.lstm(x, hc)

In [118]:
# Define input size, hidden size, sequence length, and batch size
input_size = 6
hidden_size = 4
seq_len = 5
batch_size = 5

factor = 0.01
reverse = True
#inputs = np.ones((batch_size, seq_len, input_size)).astype(np.float64)*factor
inputs = np.reshape(np.arange(seq_len*input_size*batch_size), (batch_size, seq_len , input_size)).astype(np.float64)*factor
print(inputs.reshape(batch_size, seq_len*input_size))
# Initialize LSTM weights and biases
w_ih = (np.reshape(np.arange(input_size*hidden_size*4), (input_size, hidden_size*4)).T.astype(np.float64) - (input_size + hidden_size)*hidden_size)*factor
w_hh = (np.reshape(np.arange(input_size*hidden_size*4, (input_size + hidden_size)*hidden_size*4), (hidden_size,hidden_size*4)).T.astype(np.float64) - (input_size + hidden_size)*hidden_size)*factor
print(np.concat([w_ih, w_hh], axis=1).T)
indices = [i for i in range(hidden_size*2)] + [i for i in range(hidden_size*3, hidden_size*4)]+ [i for i in range(hidden_size*2, hidden_size*3)]
w_ih = w_ih[indices,:]
w_hh = w_hh[indices,:]

b_ih = np.ones((4 * hidden_size)).astype(np.float64)*factor
b_hh = np.zeros((4 * hidden_size)).astype(np.float64)*factor

# Initialize hidden state and cell state
h0 = np.ones((2 if reverse else 1, batch_size, hidden_size)).astype(np.float64)*factor
c0 = np.zeros((2 if reverse else 1, batch_size, hidden_size)).astype(np.float64)*factor

pytorch_lstm_layer = CustomLSTM(input_size, hidden_size, reverse=reverse)
inputs_torch = torch.from_numpy(inputs)
h0_torch = torch.from_numpy(h0)
c0_torch = torch.from_numpy(c0)

# Enable gradient tracking
inputs_torch.requires_grad = True
h0_torch.requires_grad = True
c0_torch.requires_grad = True

pytorch_output, (pytorch_h_n, pytorch_c_n) = pytorch_lstm_layer(inputs_torch, (h0_torch, c0_torch))
pytorch_output = pytorch_output.reshape((batch_size, hidden_size*2*seq_len))
pytorch_output.detach().numpy()


[[0.   0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1  0.11 0.12 0.13
  0.14 0.15 0.16 0.17 0.18 0.19 0.2  0.21 0.22 0.23 0.24 0.25 0.26 0.27
  0.28 0.29]
 [0.3  0.31 0.32 0.33 0.34 0.35 0.36 0.37 0.38 0.39 0.4  0.41 0.42 0.43
  0.44 0.45 0.46 0.47 0.48 0.49 0.5  0.51 0.52 0.53 0.54 0.55 0.56 0.57
  0.58 0.59]
 [0.6  0.61 0.62 0.63 0.64 0.65 0.66 0.67 0.68 0.69 0.7  0.71 0.72 0.73
  0.74 0.75 0.76 0.77 0.78 0.79 0.8  0.81 0.82 0.83 0.84 0.85 0.86 0.87
  0.88 0.89]
 [0.9  0.91 0.92 0.93 0.94 0.95 0.96 0.97 0.98 0.99 1.   1.01 1.02 1.03
  1.04 1.05 1.06 1.07 1.08 1.09 1.1  1.11 1.12 1.13 1.14 1.15 1.16 1.17
  1.18 1.19]
 [1.2  1.21 1.22 1.23 1.24 1.25 1.26 1.27 1.28 1.29 1.3  1.31 1.32 1.33
  1.34 1.35 1.36 1.37 1.38 1.39 1.4  1.41 1.42 1.43 1.44 1.45 1.46 1.47
  1.48 1.49]]
[[-0.4  -0.39 -0.38 -0.37 -0.36 -0.35 -0.34 -0.33 -0.32 -0.31 -0.3  -0.29
  -0.28 -0.27 -0.26 -0.25]
 [-0.24 -0.23 -0.22 -0.21 -0.2  -0.19 -0.18 -0.17 -0.16 -0.15 -0.14 -0.13
  -0.12 -0.11 -0.1  -0.09]
 [-0.08 -0.0

array([[0.02494327, 0.02549661, 0.02605162, 0.02660827, 0.91415077,
        0.9171717 , 0.92008158, 0.92288446, 0.06879361, 0.07115313,
        0.07352991, 0.07592374, 0.74173725, 0.74889057, 0.7558509 ,
        0.76262132, 0.16891878, 0.17482741, 0.18077169, 0.18674926,
        0.43701335, 0.44759512, 0.45806633, 0.46842031, 0.3868539 ,
        0.39716924, 0.40740063, 0.41754241, 0.19865817, 0.20780044,
        0.21699948, 0.22624692, 0.70759084, 0.71699797, 0.72611636,
        0.73495229, 0.07341666, 0.07881886, 0.08430635, 0.08987541],
       [0.0856084 , 0.09232973, 0.09916359, 0.106103  , 0.96435869,
        0.96645644, 0.96842296, 0.97026709, 0.26340915, 0.27753663,
        0.29166843, 0.30578039, 0.91811008, 0.92299839, 0.92757362,
        0.93185579, 0.58503545, 0.60064415, 0.6157697 , 0.63041019,
        0.725904  , 0.73908866, 0.75165258, 0.76361264, 0.86529466,
        0.8730929 , 0.88042822, 0.8873266 , 0.38522593, 0.40393281,
        0.42234847, 0.44043511, 0.95750631, 0.9

In [121]:
# Define a dummy gradient (dout)
dout = np.reshape(np.arange(0, pytorch_output.nelement()), pytorch_output.shape) #torch.tensor(pytorch_output.numpy()) #torch.ones_like(pytorch_output)
dout = torch.tensor(dout, dtype=torch.float64)
# Perform backward pass
pytorch_output.backward(dout)

# Retrieve gradients
dX = inputs_torch.grad
dh0 = h0_torch.grad
dc0 = c0_torch.grad
dW = pytorch_lstm_layer.lstm.weight_ih_l0.grad
dB = pytorch_lstm_layer.lstm.bias_ih_l0.grad
dW_hh = pytorch_lstm_layer.lstm.weight_hh_l0.grad
dB_hh = pytorch_lstm_layer.lstm.bias_hh_l0.grad

dW = dW[indices,:]
dW_hh = dW_hh[indices,:]
dB = dB[indices]
dB_hh = dB_hh[indices]
if reverse:
    dW_reverse = pytorch_lstm_layer.lstm.weight_ih_l0_reverse.grad
    dB_reverse = pytorch_lstm_layer.lstm.bias_ih_l0_reverse.grad
    dW_hh_reverse = pytorch_lstm_layer.lstm.weight_hh_l0_reverse.grad
    dB_hh_reverse = pytorch_lstm_layer.lstm.bias_hh_l0_reverse.grad
    
    dW_reverse = dW_reverse[indices,:]
    dW_hh_reverse = dW_hh_reverse[indices,:]
    dB_reverse = dB_reverse[indices]
    dB_hh_reverse = dB_hh_reverse[indices]
else:
    dW_reverse, dB_reverse, dW_hh_reverse, dB_hh_reverse = None, None, None, None

#Print gradients
print("Gradient dX:\n", dX.reshape((batch_size, seq_len*input_size)))
print("Gradient dout0", dh0.reshape((2*batch_size,hidden_size)))
print("Gradient dc0", dc0.reshape((2*batch_size,hidden_size)))
print("Gradient dW:\n", torch.concat((dW.T, dW_hh.T), ))
print("Gradient dB:\n", dB)
print("Gradient dB2:\n", dB_hh)

if reverse:

    print("Gradient dW:\n", torch.concat((dW_reverse.T, dW_hh_reverse.T), ))
    print("Gradient dB:\n", dB_reverse)
    print("Gradient dB2:\n", dB_hh_reverse)


Gradient dX:
 tensor([[  -146.566,    -59.477,     27.613,    114.702,    201.791,    288.880,   -100.210,    -42.592,     15.026,     72.643,    130.261,    187.878,    -72.124,    -32.764,      6.595,     45.955,     85.314,    124.674,    -68.747,    -31.885,      4.976,     41.837,     78.699,    115.560,    -90.536,    -39.476,     11.584,     62.644,    113.705,    164.765],
        [  -146.492,    -63.741,     19.009,    101.760,    184.510,    267.261,    -83.557,    -39.337,      4.884,     49.104,     93.324,    137.544,    -47.340,    -23.818,     -0.295,     23.227,     46.749,     70.271,    -68.389,    -33.507,      1.374,     36.256,     71.138,    106.020,   -134.583,    -61.182,     12.220,     85.622,    159.024,    232.426],
        [  -151.859,    -69.654,     12.551,     94.757,    176.962,    259.167,    -73.233,    -36.514,      0.205,     36.924,     73.643,    110.362,    -33.894,    -17.178,     -0.461,     16.256,     32.973,     49.689,    -64.225,    -32.57

In [124]:
out1 = torch.concat((dW.T, dW_hh.T, dW_reverse.T, dW_hh_reverse.T, dB.reshape(1,-1), dB_reverse.reshape(1,-1)))
out2 = torch.concat((dh0.reshape((2*batch_size,hidden_size)), dc0.reshape((2*batch_size,hidden_size))))
out3 = dX.reshape((batch_size, seq_len*input_size))
out3.numpy()

array([[-1.46566052e+02, -5.94767652e+01,  2.76125218e+01,
         1.14701809e+02,  2.01791096e+02,  2.88880383e+02,
        -1.00209715e+02, -4.25920814e+01,  1.50255519e+01,
         7.26431852e+01,  1.30260819e+02,  1.87878452e+02,
        -7.21240325e+01, -3.27644974e+01,  6.59503771e+00,
         4.59545728e+01,  8.53141080e+01,  1.24673643e+02,
        -6.87467411e+01, -3.18854260e+01,  4.97588908e+00,
         4.18372042e+01,  7.86985193e+01,  1.15559834e+02,
        -9.05361451e+01, -3.94759533e+01,  1.15842386e+01,
         6.26444304e+01,  1.13704622e+02,  1.64764814e+02],
       [-1.46491915e+02, -6.37413140e+01,  1.90092867e+01,
         1.01759887e+02,  1.84510488e+02,  2.67261089e+02,
        -8.35569272e+01, -3.93367126e+01,  4.88350205e+00,
         4.91037167e+01,  9.33239313e+01,  1.37544146e+02,
        -4.73398356e+01, -2.38176031e+01, -2.95370585e-01,
         2.32268619e+01,  4.67490945e+01,  7.02713270e+01,
        -6.83891381e+01, -3.35073671e+01,  1.37440399e+

In [5]:
a = ""
for i in range(seq_len*input_size):
    a += "{} ".format(str(i))
a[:-1]

'0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199'

In [100]:
torch.tanh(torch.tensor(0.5))

tensor(0.4621)

In [101]:
torch.sigmoid(torch.tensor(0.5))

tensor(0.6225)